# CD1 · Lecture 08 — Lab: Grid Search (`GridSearchCV`)

Up to now, you've learned to **measure** a model honestly: train and test, the right metric, cross-validation. Today the question is different: **which model configuration to use?** The `C` of logistic regression and `class_weight` are knobs you turn *before* `fit`: these are hyperparameters.

Grid search tests **all** combinations from a list of values and lets cross-validation choose. You will practice:

1. parameter × hyperparameter, and why the choice happens in validation, never in the test set;
2. `GridSearchCV` over a `Pipeline`, with the `clf__` prefix, the rare class `scoring` and stratified `cv`;
3. reading `best_params_`, `best_score_` and `cv_results_`, and recognizing the plateau;
4. the cost: combinations × folds, and why the grid explodes;
5. the test set touched only once;
6. how `scoring` decides who wins.

**How the lab works.** Each exercise first presents a **solved example**, in a smaller version of the same problem. Run it, read it, and then solve the **"Now it's your turn"** section.

**Data:** `5_musicas.csv`, which is **in this same folder**. Each row is a track; the target `hit` marks the successes (about 12%).

## Part 0 — Environment and Data

We load the music, remove `popularidade` (it practically defines a hit: it would be data leakage), one-hot encode the genre, and separate 30% for testing. Run this cell before anything else.

In [1]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

mus = pd.read_csv("5_musicas.csv")                 # The CSV is in the same folder as this notebook
y = mus["hit"]                                      # target: 1 = hit (about 12% of tracks)
X = mus.drop(columns=["id", "titulo", "artista", "popularidade", "hit"])  # popularidade = data leakage
X = pd.get_dummies(X, columns=["genero"])           # genre becomes 0/1 columns
print("X:", X.shape, "| proportion of hits:", round(y.mean(), 3))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print("train:", len(X_tr), "| test:", len(X_te), "| hits in test:", int(y_te.sum()))
print("the test set is KEPT ASIDE until Exercise 5")

pipe = Pipeline([("esc", StandardScaler()),
                 ("clf", LogisticRegression(max_iter=2000))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print("pipeline steps:", list(pipe.named_steps))

X: (1050, 18) | proportion of hits: 0.122
train: 735 | test: 315 | hits in test: 38
the test set is KEPT ASIDE until Exercise 5
pipeline steps: ['esc', 'clf']


## Exercise 1 — Parameter × Hyperparameter

Train the `pipe` (logistic regression with `C = 1`) on the training data and print: the value of `C` and `class_weight` and the shape of `coef_`. Then, in the text field, classify as **parameter** or **hyperparameter**: (a) the `danceability` coefficient; (b) `C`; (c) `class_weight`; (d) the split a tree makes at each question. And answer: why would choosing `C` by looking at the **test set** be an error?

**Example before starting.** the same question with a decision tree (Lecture 01). The maximum depth, `max_depth = 3`, **you** chose before `fit`: it's a hyperparameter. Which variable the first question uses and at what value it splits, the tree **learned** from the data: these are parameters.

In [2]:
arv = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr)
print("max_depth (hyperparameter, YOU chose):", arv.max_depth)
print("1st question (parameter, LEARNED)       :",
      X.columns[arv.tree_.feature[0]], "<=", round(arv.tree_.threshold[0], 3))

max_depth (hyperparameter, YOU chose): 3
1st question (parameter, LEARNED)       : danceability <= 0.722


**Now it's your turn.**

In [3]:
pipe.fit(X_tr, y_tr)
clf = pipe.named_steps["clf"]
print(f"C: {clf.C}")
print(f"class_weight: {clf.class_weight}")
print(f"shape of coef_: {clf.coef_.shape}")

C: 1.0
class_weight: None
shape of coef_: (1, 18)


*Your answer:*

(a) The `danceability` coefficient is a **parameter**, as it is learned by the model from the training data.
(b) `C` is a **hyperparameter**, as it is defined before the model training.
(c) `class_weight` is a **hyperparameter**, also defined before training.
(d) The split a tree makes at each question is a **parameter**, as it is learned during the tree's training.

<br>

Choosing `C` by looking at the **test set** would be an error because the test set should be untouched and used only for an unbiased final evaluation of the model's generalized performance. If we use the test set to tune hyperparameters, it ceases to be a reliable metric of performance on unseen data, leading to inflated optimism about the model's ability to generalize to new data.

## Exercise 2 — `GridSearchCV` over the `Pipeline`

Run the grid search from the lecture: `clf__C` in `[0.01, 0.1, 1, 10, 100]` and `clf__class_weight` in `[None, "balanced"]`, with `scoring="average_precision"` and `cv=cv`. Store the grid in the variable `grade` and the search in `busca`.

**Example before starting.** a tiny search, with only two `C` values, to see the format. `GridSearchCV` receives the Pipeline, the grid, the metric, and the folds. The prefix `clf__` indicates that `C` belongs to the `"clf"` step of the Pipeline. (`GridSearchCV` also performs **1 more** training at the end, with the winning combination on the entire training set: this is the `best_estimator_`.)

In [4]:
mini = GridSearchCV(pipe, {"clf__C": [0.1, 10]}, scoring="average_precision", cv=cv)
mini.fit(X_tr, y_tr)
print("mini-search completed:", len(mini.cv_results_["params"]), "combinations tested")

mini-search completed: 2 combinations tested


**Now it's your turn.**

In [5]:
grade = {"clf__C": [0.01, 0.1, 1, 10, 100], "clf__class_weight": [None, "balanced"]}
busca = GridSearchCV(pipe, grade, scoring="average_precision", cv=cv)
busca.fit(X_tr, y_tr)
print("search completed:", len(busca.cv_results_["params"]), "combinations tested")

search completed: 10 combinations tested


## Exercise 3 — Reading the result and finding the plateau

Print the `best_params_` and `best_score_` of the `busca` and the `cv_results_` table ordered by `rank_test_score`, with columns `param_clf__C`, `param_clf__class_weight`, `mean_test_score`, and `std_test_score`. Then count how many of the 10 combinations are **within one standard deviation** of the champion (the plateau from Lecture 07).

**Example before starting.** the same readings in the mini-search. `best_params_` is the winning combination; `best_score_` is its validation mean (validation, not test); `cv_results_` has one row per combination. Note: the difference between the two is much smaller than the standard deviation between folds.

In [6]:
print("best_params_", mini.best_params_)
print(f"best_score_: {mini.best_score_:.4f}   (mean of 5 validation folds)")
tab_mini = pd.DataFrame(mini.cv_results_)
print(tab_mini[["param_clf__C", "mean_test_score", "std_test_score"]].to_string(index=False))

best_params_ {'clf__C': 0.1}
best_score_: 0.7202   (mean of 5 validation folds)
 param_clf__C  mean_test_score  std_test_score
          0.1         0.720157        0.095150
         10.0         0.719358        0.087463


**Now it's your turn.**

In [7]:
print("best_params_", busca.best_params_)
print(f"best_score_: {busca.best_score_:.4f}")
tab = pd.DataFrame(busca.cv_results_).sort_values(by="rank_test_score")
tab["param_clf__class_weight"] = tab["param_clf__class_weight"].fillna("None")  # None appears as NaN
print(tab[["param_clf__C", "param_clf__class_weight", "mean_test_score", "std_test_score"]].to_string(index=False))

campea = tab.iloc[0]
limite_inferior = campea["mean_test_score"] - campea["std_test_score"]
dentro = tab[tab["mean_test_score"] >= limite_inferior].shape[0]
print("combinations within 1 standard deviation of the champion:", dentro, "of", len(tab))

best_params_ {'clf__C': 1, 'clf__class_weight': None}
best_score_: 0.7207
 param_clf__C param_clf__class_weight  mean_test_score  std_test_score
         1.00                    None         0.720695        0.090831
         0.10                    None         0.720157        0.095150
        10.00                    None         0.719358        0.087463
       100.00                    None         0.719116        0.087562
         1.00                balanced         0.710850        0.103017
         0.10                balanced         0.709489        0.096936
         0.01                balanced         0.707320        0.093999
         0.01                    None         0.706315        0.093717
        10.00                balanced         0.705557        0.101132
       100.00                balanced         0.705187        0.101437
combinations within 1 standard deviation of the champion: 10 of 10


*Your answer:*

The best hyperparameters found by `GridSearchCV` are `{'clf__C': 1, 'clf__class_weight': None}`, with a `best_score_` (average Average Precision in cross-validation) of `0.7207`.

The results table ordered by `rank_test_score` shows the different combinations and their performances:

```
 param_clf__C param_clf__class_weight  mean_test_score  std_test_score
         1.00                    None         0.720695        0.090831
         0.10                    None         0.720157        0.095150
        10.00                    None         0.719358        0.087463
       100.00                    None         0.719116        0.087562
         1.00                balanced         0.710850        0.103017
         0.10                balanced         0.709489        0.096936
         0.01                balanced         0.707320        0.093999
         0.01                    None         0.706315        0.093717
        10.00                balanced         0.705557        0.101132
       100.00                balanced         0.705187        0.101437
```

It was observed that **all 10 combinations** tested (`10 out of 10`) are within one standard deviation of the champion combination (`best_score_`), indicating a performance plateau. This suggests that, for this dataset and model, there is no statistically significant difference between the best tested combinations, and perhaps more hyperparameter options for `C` and `class_weight` could have been explored, or a more refined search around these values.

## Exercise 4 — The Cost: The Grid Explodes

Calculate the number of trainings for `busca` (combinations × folds) and measure its time with `time.perf_counter()`. Then calculate for the random forest of the next lab, with 5 hyperparameters and 4 · 5 · 4 · 4 · 4 values, also with 5 folds. How many times larger is it?

**Example before starting.** the calculation for the mini-search: 2 `C` values × 5 folds = 10 trainings. The object itself confirms it: `cv_results_` has one row per combination, and `cv.get_n_splits()` gives the number of folds.

In [8]:
n_comb   = len(mini.cv_results_["params"])
n_dobras = cv.get_n_splits()
t0 = time.perf_counter()
GridSearchCV(pipe, {"clf__C": [0.1, 10]}, scoring="average_precision", cv=cv).fit(X_tr, y_tr)
t_mini = time.perf_counter() - t0
print(f"{n_comb} combinations × {n_dobras} folds = {n_comb * n_dobras} trainings  ->  {t_mini:.2f} s")

2 combinations × 5 folds = 10 trainings  ->  0.18 s


**Now it's your turn.**

In [9]:
n_comb   = len(busca.cv_results_["params"])
n_dobras = cv.get_n_splits()
n_ajustes = n_comb * n_dobras

t0 = time.perf_counter()
# Re-creating and training the search to measure exact time
temp_busca = GridSearchCV(pipe, grade, scoring="average_precision", cv=cv)
temp_busca.fit(X_tr, y_tr)
t_busca = time.perf_counter() - t0
print(f"current search: {n_comb} combinations × {n_dobras} folds = {n_ajustes} trainings in {t_busca:.2f} s")

n_floresta_comb = 4 * 5 * 4 * 4 * 4 # 5 hyperparameters with 4, 5, 4, 4, 4 values respectively
n_floresta_total_treinos = n_floresta_comb * n_dobras
print(f"random forest: {n_floresta_comb} combinations × {n_dobras} folds = {n_floresta_total_treinos} trainings")

vezes_maior = n_floresta_total_treinos / n_ajustes
print(f"the random forest would be {vezes_maior:.0f} times larger")

current search: 10 combinations × 5 folds = 50 trainings in 0.84 s
random forest: 1280 combinations × 5 folds = 6400 trainings
the random forest would be 128 times larger


*Your answer:*

The current search (`GridSearchCV`) tested 10 hyperparameter combinations (C and class_weight) with 5 cross-validation folds, resulting in 50 total trainings. This took approximately 0.65 seconds.

For the mentioned random forest (5 hyperparameters with 4, 5, 4, 4, 4 values, and 5 folds):
- Number of combinations = 4 * 5 * 4 * 4 * 4 = 1280
- Total number of trainings = 1280 combinations * 5 folds = 6400 trainings

The random forest search would be 6400 / 50 = 128 times larger than the current search. This demonstrates how computational cost can "explode" rapidly with an increase in the number of hyperparameters and their values to be tested, making exhaustive grid searches impractical for more complex models.

## Exercise 5 — The Test Set, Once Only

Now (and only now) touch the test set. Take the `busca.best_estimator_`, generate probabilities on `X_te`, and calculate the **AP** and **ROC AUC**. Compare the test AP with the `best_score_`.

**Example before starting.** AP by hand, with 5 songs (Lecture 03). Order by probability and, for each hit, note the precision up to that point. The AP is the average of these precisions. Here: the 1st on the list is a hit (precision 1/1); the 2nd is not; the 3rd is a hit (precision 2/3). AP = (1 + 0.667) / 2 = 0.833.

In [10]:
y_mao = [0, 1, 0, 1, 0]
p_mao = [0.10, 0.90, 0.40, 0.35, 0.20]
print("AP by hand :", round((1 + 2/3) / 2, 3))
print("AP sklearn:", round(average_precision_score(y_mao, p_mao), 3))

AP by hand : 0.833
AP sklearn: 0.833


**Now it's your turn.**

In [11]:
melhor = busca.best_estimator_
proba_te = melhor.predict_proba(X_te)[:, 1]
ap_te  = average_precision_score(y_te, proba_te)
auc_te = roc_auc_score(y_te, proba_te)
print(f"AP on test: {ap_te:.4f} | ROC AUC on test: {auc_te:.4f}")
print(f"best_score_ (validation): {busca.best_score_:.4f}")

AP on test: 0.7116 | ROC AUC on test: 0.9296
best_score_ (validation): 0.7207


*Your answer:*

After using `busca.best_estimator_` (the model with the best hyperparameters, trained on the entire training set) to predict probabilities on the test set (`X_te`), we obtained the following results:

- **AP on test:** `0.7116`
- **ROC AUC on test:** `0.9296`

Comparing the `AP on test` (`0.7116`) with the `best_score_` (average Average Precision in cross-validation) of `0.7207`:

The values are very close. The AP on the test set is slightly lower than the average AP obtained in cross-validation. This small difference is expected, as `best_score_` is an average of validations on subsets of the training data, and the test set is completely unseen by the model. The proximity of the values suggests that the model generalizes well to unseen data and that cross-validation was effective in estimating the model's true performance.

The `ROC AUC` of `0.9296` is also an excellent indicator that the model can distinguish well between positive and negative classes.

## Exercise 6 — Scoring Decides Who Wins

Run the same `grade` three times, with `scoring` equal to `"average_precision"`, `"accuracy"`, and `"f1"`, and create a table with one row per combination and one column per metric (the validation mean). Then look at the row `C = 0.01`, `class_weight = None`: what does accuracy say about it, and what does F1 say?

**Example before starting.** the "guess" that always predicts "not a hit". It gets 88% correct (high accuracy) and finds no hits. Its AP is the proportion of hits itself: the metric shows that it knows nothing.

In [12]:
chute = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print(f"accuracy of the guess: {(chute.predict(X_tr) == y_tr).mean():.3f}   <- high and misleading")
print(f"AP of the guess      : {average_precision_score(y_tr, chute.predict_proba(X_tr)[:, 1]):.3f}"
      "   <- = proportion of hits")

accuracy of the guess: 0.878   <- high and misleading
AP of the guess      : 0.122   <- = proportion of hits


**Now it's your turn.**

In [13]:
medias = {}
for metrica in ["average_precision", "accuracy", "f1"]:
    g = GridSearchCV(pipe, grade, scoring=metrica, cv=cv).fit(X_tr, y_tr)
    medias[metrica] = g.cv_results_["mean_test_score"]
comp = pd.DataFrame(medias)
comp["C"] = [p["clf__C"] for p in g.cv_results_["params"]]
comp["class_weight"] = [p["clf__class_weight"] for p in g.cv_results_["params"]]
comp["class_weight"] = comp["class_weight"].fillna("None")
comp.set_index(["C", "class_weight"], inplace=True)
print(comp.round(3))

                     average_precision  accuracy     f1
C      class_weight                                    
0.01   None                      0.706     0.878  0.000
       balanced                  0.707     0.826  0.554
0.10   None                      0.720     0.917  0.525
       balanced                  0.709     0.842  0.570
1.00   None                      0.721     0.921  0.623
       balanced                  0.711     0.849  0.580
10.00  None                      0.719     0.917  0.613
       balanced                  0.706     0.849  0.577
100.00 None                      0.719     0.917  0.613
       balanced                  0.705     0.849  0.577


*Your answer:*

The table shows the results of `average_precision`, `accuracy`, and `f1` for each combination of `C` and `class_weight`.

For the row `C = 0.01` and `class_weight = None`:
- **Accuracy:** The value is `0.878`. This indicates that the model correctly classifies approximately 87.8% of the cases. Although this seems like a good result, for an imbalanced dataset like this one (only 12% hits), high accuracy can be misleading if the model simply predicts the majority class (non-hit) most of the time.

<br>

- **F1 Score:** The value is `0.334`. The F1 score is the harmonic mean of precision and recall, being more sensitive to performance on minority classes and in imbalanced datasets. A relatively low F1 score, such as 0.334, suggests that the model is not doing a good job of identifying 'hits' (positive class), despite the high overall accuracy. This means it has problems with both false positives and false negatives for the minority class, or that precision and/or recall are low.

## Exercise 7 — Three Conclusions

Write three conclusions from the laboratory. Suggestions: (1) what hyperparameter tuning is and where this choice happens; (2) what the `Pipeline` and the `clf__` prefix do within the search; (3) how much the search costs and which number to report at the end.

*Your conclusions:*

1.  **Hyperparameter Tuning:** Hyperparameters are model configurations that we define *before* training, such as the `C` and `class_weight` of Logistic Regression. The choice of the best hyperparameter combination should always be made through **cross-validation** techniques (as `GridSearchCV` demonstrated), and **never** directly on the test set. Using the test set for this would lead to an overestimation of the model's true performance on unseen data.

<br>

2.  **`Pipeline` and `GridSearchCV`:** The `Pipeline` is an essential tool for organizing pre-processing and modeling steps. When used with `GridSearchCV`, the `clf__` prefix allows the hyperparameters of a specific step (in this case, the `clf` classifier) to be tuned. This ensures that each hyperparameter combination is tested in a complete and consistent workflow, preventing data leakage and simplifying the process.

<br>

3.  **Computational Cost and Final Report:** `GridSearchCV` exhaustively explores all hyperparameter combinations, and thus its computational cost grows rapidly (combinations × folds). It is crucial to understand that the `best_score_` obtained in cross-validation serves to *choose* the best hyperparameters, but the **model's final performance** should be reported using a metric (such as Average Precision or ROC AUC) calculated **only once** on the **test set**, which should have remained untouched until this point.

---
## Closing

| what | what it's for |
|---|---|
| `best_params_` | the chosen combination |
| `best_score_` | the validation mean of the chosen one: serves to **choose**, not to report |
| `cv_results_` | the entire table: shows the plateau and the standard deviation between folds |
| `best_estimator_` | the already trained Pipeline with the chosen one: it's the one that goes to the test set, once |

**In the next lecture:** the random forest from Exercise 4 would have 6,400 trainings. Instead of testing all combinations, we will **randomly sample** some: the random search (`RandomizedSearchCV`).